# Daily Challenge - Week 14: BERT Attention, Fine-Tuning & Explainable Inference

Ce notebook consolide les concepts de la semaine :
- Tokenisation
- Cartes d'attention (attention maps)
- Fine-tuning de transformers
- Évaluation et inférence expliquable

**Dataset** : `tweet_eval` (configuration `sentiment`) — 3 classes : négatif, neutre, positif  
**Modèle** : `distilbert-base-uncased`

In [ ]:
# Installation des dépendances
!pip install transformers datasets evaluate torch matplotlib seaborn scikit-learn --quiet

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel,
    TrainingArguments,
    Trainer,
)
import evaluate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")

---
## Partie 1 — Chargement et inspection des données

In [ ]:
# Chargement du dataset tweet_eval (sentiment)
dataset = load_dataset("tweet_eval", "sentiment")
print(dataset)

# Mapping des labels
LABEL_NAMES = ["negative", "neutral", "positive"]
print("\nLabels disponibles :", LABEL_NAMES)

In [ ]:
# Distribution des classes dans chaque split
for split in ["train", "validation", "test"]:
    labels = dataset[split]["label"]
    counts = Counter(labels)
    print(f"\n{split.upper()} — {len(labels)} exemples")
    for idx, name in enumerate(LABEL_NAMES):
        print(f"  {name} ({idx}) : {counts[idx]}")

In [ ]:
# Sauvegarde de 2 exemples par label pour la visualisation
example_tweets = {label: [] for label in LABEL_NAMES}

for item in dataset["train"]:
    label_name = LABEL_NAMES[item["label"]]
    if len(example_tweets[label_name]) < 2:
        example_tweets[label_name].append(item["text"])
    if all(len(v) == 2 for v in example_tweets.values()):
        break

print("Exemples sauvegardés :")
for label, tweets in example_tweets.items():
    print(f"\n[{label.upper()}]")
    for t in tweets:
        print(f"  - {t}")

---
## Partie 2 — Pipeline de tokenisation

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(batch):
    """Tokenise les tweets : troncature/padding à 128 tokens."""
    encoded = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128,
    )
    encoded["labels"] = batch["label"]
    return encoded

# Application du prétraitement
tokenized = dataset.map(preprocess, batched=True)
tokenized = tokenized.shuffle(seed=42)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

print("Colonnes disponibles :", tokenized["train"].column_names)
print("Exemple de forme input_ids :", tokenized["train"][0]["input_ids"].shape)

---
## Partie 3 — Fine-tuning de DistilBERT

In [ ]:
# Chargement du modèle pour classification (3 labels)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label={i: l for i, l in enumerate(LABEL_NAMES)},
    label2id={l: i for i, l in enumerate(LABEL_NAMES)},
)
model = model.to(DEVICE)
print("Modèle chargé.")

In [ ]:
# Métriques d'évaluation
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="macro")
    return {"accuracy": acc["accuracy"], "f1_macro": f1["f1"]}

In [ ]:
training_args = TrainingArguments(
    output_dir="./distilbert-tweet-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=5e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=50,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=compute_metrics,
)

print("Lancement de l'entraînement...")
trainer.train()

In [ ]:
# Sauvegarde du meilleur checkpoint
trainer.save_model("./distilbert-tweet-sentiment/best")
tokenizer.save_pretrained("./distilbert-tweet-sentiment/best")
print("Modèle et tokenizer sauvegardés dans ./distilbert-tweet-sentiment/best")

---
## Partie 4 — Évaluation & Histogramme de confiance

In [ ]:
# Évaluation sur le split validation
val_results = trainer.evaluate(tokenized["validation"])
print("=== Résultats Validation ===")
print(f"  Accuracy  : {val_results['eval_accuracy']:.4f}")
print(f"  Macro F1  : {val_results['eval_f1_macro']:.4f}")

In [ ]:
# Collecte des scores softmax sur le split test
test_outputs = trainer.predict(tokenized["test"])
logits = test_outputs.predictions
probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()

# Score de confiance = probabilité de la classe prédite
predicted_classes = np.argmax(probs, axis=-1)
confidence_scores = probs[np.arange(len(probs)), predicted_classes]

print(f"Confiance moyenne : {confidence_scores.mean():.4f}")
print(f"Confiance médiane : {np.median(confidence_scores):.4f}")

In [ ]:
# Histogramme des scores de confiance
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(confidence_scores, bins=10, range=(0, 1), color="steelblue", edgecolor="white", linewidth=0.8)
ax.set_xlabel("Score de confiance (softmax max)", fontsize=13)
ax.set_ylabel("Nombre de prédictions", fontsize=13)
ax.set_title("Distribution des scores de confiance — DistilBERT tweet_eval", fontsize=14)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.axvline(confidence_scores.mean(), color="tomato", linestyle="--", label=f"Moyenne = {confidence_scores.mean():.2f}")
ax.legend()
plt.tight_layout()
plt.savefig("confidence_histogram.png", dpi=150)
plt.show()
print("Histogramme sauvegardé : confidence_histogram.png")

### Analyse de l'histogramme

- **Sur-confiance** : si la majorité des scores est concentrée dans le bin [0.9–1.0], le modèle est sur-confiant — ses probabilités dépassent sa vraie précision.
- **Sous-confiance** : si les scores sont étalés uniformément, le modèle manque de discrimination.
- **Idéal** : une distribution légèrement concentrée à droite (> 0.7) avec quelques cas incertains (0.3–0.6) indique un bon calibrage.

---
## Partie 5 — Inspection des cartes d'attention

In [ ]:
# Chargement du modèle de base pour extraire les poids d'attention
attention_model = AutoModel.from_pretrained(
    "./distilbert-tweet-sentiment/best",
    output_attentions=True,
).to(DEVICE)
attention_model.eval()

# Tweet d'exemple (négatif)
EXAMPLE_TWEET = example_tweets["negative"][0]
print(f"Tweet analysé : {EXAMPLE_TWEET}")

In [ ]:
# Tokenisation du tweet d'exemple
inputs = tokenizer(
    EXAMPLE_TWEET,
    return_tensors="pt",
    truncation=True,
    max_length=128,
).to(DEVICE)

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print("Tokens :", tokens)

In [ ]:
# Extraction des poids d'attention
with torch.no_grad():
    outputs = attention_model(**inputs)

# outputs.attentions : tuple de (num_layers,) tenseurs de forme (batch, heads, seq, seq)
# Dernière couche, moyenne sur les têtes, ligne [CLS] (index 0)
last_layer_attn = outputs.attentions[-1]          # (1, heads, seq, seq)
avg_attn = last_layer_attn[0].mean(dim=0)         # (seq, seq)
cls_attn = avg_attn[0].cpu().numpy()              # (seq,) — attention du [CLS] vers chaque token

print("Forme attention (dernier layer, moyenne têtes) :", avg_attn.shape)
print("Attention [CLS] :", cls_attn)

In [ ]:
# Visualisation : heatmap d'attention [CLS] → tokens
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Bar chart ---
ax1 = axes[0]
colors = ["tomato" if a > np.percentile(cls_attn, 75) else "steelblue" for a in cls_attn]
ax1.bar(range(len(tokens)), cls_attn, color=colors)
ax1.set_xticks(range(len(tokens)))
ax1.set_xticklabels(tokens, rotation=45, ha="right", fontsize=9)
ax1.set_ylabel("Score d'attention")
ax1.set_title("Attention [CLS] → tokens (dernier layer, moy. têtes)")

# --- Heatmap (matrice complète dernier layer) ---
ax2 = axes[1]
sns.heatmap(
    avg_attn.cpu().numpy(),
    xticklabels=tokens,
    yticklabels=tokens,
    cmap="Blues",
    ax=ax2,
    linewidths=0.3,
)
ax2.set_title("Matrice d'attention complète (dernier layer)")
ax2.tick_params(axis="x", rotation=45, labelsize=8)
ax2.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig("attention_heatmap.png", dpi=150)
plt.show()
print("Heatmap sauvegardée : attention_heatmap.png")

In [ ]:
# Top 5 tokens les plus attendus par [CLS]
top5_indices = np.argsort(cls_attn)[::-1][:5]
print("Top 5 tokens (hors [CLS]/[SEP]/[PAD]) :")
for idx in top5_indices:
    tok = tokens[idx]
    if tok not in ["[CLS]", "[SEP]", "[PAD]"]:
        print(f"  '{tok}' — score : {cls_attn[idx]:.4f}")

---
## Partie 6 — Fonction d'inférence expliquable `analyze_text()`

In [ ]:
from torch.nn.functional import softmax as torch_softmax

# Rechargement propre du modèle de classification
inference_model = AutoModelForSequenceClassification.from_pretrained(
    "./distilbert-tweet-sentiment/best",
    output_attentions=True,
).to(DEVICE)
inference_model.eval()


def analyze_text(text: str, top_k: int = 5) -> dict:
    """
    Analyse un texte et retourne :
      - label      : classe prédite (negative / neutral / positive)
      - confidence : score softmax de la classe prédite
      - highlighted_tokens : top_k tokens auxquels [CLS] prête le plus d'attention

    Parameters
    ----------
    text : str
        Texte à analyser.
    top_k : int
        Nombre de tokens saillants à retourner.

    Returns
    -------
    dict with keys: label, confidence, highlighted_tokens
    """
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
    ).to(DEVICE)

    with torch.no_grad():
        outputs = inference_model(**inputs)

    # Prédiction
    logits = outputs.logits[0]                        # (num_labels,)
    probs = torch_softmax(logits, dim=-1).cpu()       # (num_labels,)
    predicted_idx = int(probs.argmax())
    confidence = float(probs[predicted_idx])
    label = LABEL_NAMES[predicted_idx]

    # Tokens saillants via attention [CLS]
    last_attn = outputs.attentions[-1][0].mean(dim=0)  # (seq, seq)
    cls_attn_scores = last_attn[0].cpu().numpy()       # (seq,)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Filtrer les tokens spéciaux
    special = {"[CLS]", "[SEP]", "[PAD]"}
    ranked = sorted(
        [(tok, score) for tok, score in zip(tokens, cls_attn_scores) if tok not in special],
        key=lambda x: x[1],
        reverse=True,
    )
    highlighted_tokens = [(tok, round(float(score), 4)) for tok, score in ranked[:top_k]]

    return {
        "label": label,
        "confidence": round(confidence, 4),
        "highlighted_tokens": highlighted_tokens,
    }

In [ ]:
# Tests de la fonction
test_sentences = [
    "This product is absolutely terrible, worst purchase ever!",
    "The package arrived on time and everything looks fine.",
    "I love this! Best thing I've bought this year, highly recommend!",
]

for sentence in test_sentences:
    result = analyze_text(sentence)
    print(f"\nTexte  : {sentence}")
    print(f"Label  : {result['label']}  (confiance : {result['confidence']:.2%})")
    print(f"Tokens saillants : {result['highlighted_tokens']}")

---
## Récapitulatif

| Section | Livrables |
|---|---|
| 1. Données | Distribution des labels, 2 exemples par classe |
| 2. Tokenisation | Prétraitement batché, format PyTorch |
| 3. Fine-tuning | DistilBERT 3 labels, 3 époques, sauvegarde meilleur checkpoint |
| 4. Évaluation | Accuracy + Macro F1, histogramme de confiance (`confidence_histogram.png`) |
| 5. Attention | Heatmap attention [CLS] dernier layer (`attention_heatmap.png`) |
| 6. Inférence | Fonction `analyze_text()` → `{label, confidence, highlighted_tokens}` |

### Artefacts sauvegardés
- `./distilbert-tweet-sentiment/best/` — modèle + tokenizer
- `confidence_histogram.png`
- `attention_heatmap.png`